MODEL A : GROUP STAGE

In [1]:
import itertools
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

# ============================================================
# 1) INPUT DATA
# ============================================================

# Groups from your document
groups = {
    "A": ["Morocco", "Zambia", "Mali", "Comoros"],
    "B": ["Angola", "South Africa", "Egypt", "Zimbabwe"],
    "C": ["Nigeria", "Tunisia", "Uganda", "Tanzania"],
    "D": ["Senegal", "Congo", "Benin", "Botswana"],
    "E": ["Algeria", "Burkina Faso", "Equatorial Guinea", "Sudan"],
    "F": ["Cote d'Ivoire", "Cameroon", "Gabon", "Mozambique"],
}

# Team base-camp cities
team_base_city = {
    "Morocco": "Rabat",
    "Zambia": "Casablanca",
    "Mali": "Casablanca",
    "Comoros": "Casablanca",
    "Angola": "Marrakesh",
    "South Africa": "Marrakesh",
    "Egypt": "Agadir",
    "Zimbabwe": "Marrakesh",
    "Nigeria": "Fez",
    "Tunisia": "Rabat",
    "Uganda": "Rabat",
    "Tanzania": "Rabat",
    "Senegal": "Tangier",
    "Congo": "Rabat",
    "Benin": "Rabat",
    "Botswana": "Rabat",
    "Algeria": "Rabat",
    "Burkina Faso": "Casablanca",
    "Equatorial Guinea": "Casablanca",
    "Sudan": "Casablanca",
    "Cote d'Ivoire": "Marrakesh",
    "Cameroon": "Agadir",
    "Gabon": "Agadir",
    "Mozambique": "Agadir",
}

# Stadiums from your document
stadiums_df = pd.DataFrame([
    {"stadium": "Grand Stadium of Agadir", "city": "Agadir", "capacity": 45480},
    {"stadium": "Mohammed V Complex", "city": "Casablanca", "capacity": 45000},
    {"stadium": "Grand Stadium of Fez", "city": "Fez", "capacity": 45000},
    {"stadium": "Grand Stadium of Marrakesh", "city": "Marrakesh", "capacity": 45240},
    {"stadium": "Grand Stadium of Tangier", "city": "Tangier", "capacity": 75600},
    {"stadium": "Prince Moulay Abdellah Sports Complex", "city": "Rabat", "capacity": 69500},
    {"stadium": "Moulay El Hassan II Stadium", "city": "Rabat", "capacity": 22000},
    {"stadium": "Stade Al Medina", "city": "Rabat", "capacity": 21000},
    {"stadium": "Rabat Olympic Stadium", "city": "Rabat", "capacity": 21000},
])

# One-way travel times in hours (city-to-city)
travel_time = {
    "Rabat": {
        "Rabat": 0.00, "Casablanca": 1.37, "Agadir": 5.57,
        "Marrakesh": 3.50, "Fez": 2.40, "Tangier": 2.83
    },
    "Casablanca": {
        "Rabat": 1.37, "Casablanca": 0.00, "Agadir": 4.80,
        "Marrakesh": 2.47, "Fez": 3.03, "Tangier": 3.77
    },
    "Agadir": {
        "Rabat": 5.57, "Casablanca": 4.80, "Agadir": 0.00,
        "Marrakesh": 2.62, "Fez": 7.27, "Tangier": 7.87
    },
    "Marrakesh": {
        "Rabat": 3.50, "Casablanca": 2.47, "Agadir": 2.62,
        "Marrakesh": 0.00, "Fez": 5.50, "Tangier": 5.90
    },
    "Fez": {
        "Rabat": 2.40, "Casablanca": 3.03, "Agadir": 7.27,
        "Marrakesh": 5.50, "Fez": 0.00, "Tangier": 4.37
    },
    "Tangier": {
        "Rabat": 2.83, "Casablanca": 3.77, "Agadir": 7.87,
        "Marrakesh": 5.90, "Fez": 4.37, "Tangier": 0.00
    },
}

# ============================================================
# 2) GENERATE THE 36 ROUND-ROBIN MATCHES
# ============================================================

matches = []
for group_name, team_list in groups.items():
    for k, (t1, t2) in enumerate(itertools.combinations(team_list, 2), start=1):
        matches.append({
            "match_id": f"{group_name}{k}",
            "group": group_name,
            "team1": t1,
            "team2": t2
        })

matches_df = pd.DataFrame(matches)

# ============================================================
# 3) DERIVED STRUCTURES
# ============================================================

teams = sorted(team_base_city.keys())
stadiums = stadiums_df["stadium"].tolist()
stadium_city = stadiums_df.set_index("stadium")["city"].to_dict()

# Indicator a_tm = 1 if team t plays in match m
a_tm = {}
for row in matches_df.itertuples(index=False):
    for t in teams:
        a_tm[(t, row.match_id)] = int(t == row.team1 or t == row.team2)

# Precompute round-trip cost for team t if match m is assigned to stadium s
# Only matters when team t is in match m
travel_cost = {}
for row in matches_df.itertuples(index=False):
    m = row.match_id
    for t in teams:
        for s in stadiums:
            if a_tm[(t, m)] == 1:
                base = team_base_city[t]
                venue_city = stadium_city[s]
                travel_cost[(t, m, s)] = 2 * travel_time[base][venue_city]
            else:
                travel_cost[(t, m, s)] = 0.0

# ============================================================
# 4) BUILD MODEL A
# ============================================================

model = gp.Model("AFCON_Model_A_Pure_Fairness")

# x[m,s] = 1 if match m assigned to stadium s
x = model.addVars(matches_df["match_id"], stadiums, vtype=GRB.BINARY, name="x")

# total travel per team
team_total = model.addVars(teams, lb=0.0, vtype=GRB.CONTINUOUS, name="team_total")

# maximum team travel
Z = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Z")

# Each match assigned to exactly one stadium
for m in matches_df["match_id"]:
    model.addConstr(gp.quicksum(x[m, s] for s in stadiums) == 1, name=f"assign_{m}")

# Team total travel definition
for t in teams:
    model.addConstr(
        team_total[t] ==
        gp.quicksum(travel_cost[(t, m, s)] * x[m, s]
                    for m in matches_df["match_id"]
                    for s in stadiums),
        name=f"team_total_{t}"
    )

# Max fairness constraint
for t in teams:
    model.addConstr(team_total[t] <= Z, name=f"max_travel_{t}")

# Objective: minimize maximum total travel across teams
model.setObjective(Z, GRB.MINIMIZE)

# ============================================================
# 5) OPTIONAL TIE-BREAKER
# ============================================================
# Model A can have many optimal solutions with the same Z.
# This second stage keeps the same optimal Z and then minimizes total travel.
# Comment this whole block out if you want pure one-stage Model A only.

model.optimize()

if model.status == GRB.OPTIMAL:
    best_Z = Z.X

    model.addConstr(Z <= best_Z + 1e-6, name="fix_best_Z")
    model.setObjective(gp.quicksum(team_total[t] for t in teams), GRB.MINIMIZE)
    model.optimize()

# ============================================================
# 6) OUTPUT ASSIGNMENT TABLE
# ============================================================

if model.status == GRB.OPTIMAL:
    assignments = []

    for row in matches_df.itertuples(index=False):
        chosen_stadium = None
        for s in stadiums:
            if x[row.match_id, s].X > 0.5:
                chosen_stadium = s
                break

        assignments.append({
            "Match ID": row.match_id,
            "Group": row.group,
            "Team 1": row.team1,
            "Team 2": row.team2,
            "Assigned Stadium": chosen_stadium,
            "Assigned City": stadium_city[chosen_stadium]
        })

    assignments_df = pd.DataFrame(assignments).sort_values(["Group", "Match ID"])
    print("\n=== MODEL A ASSIGNMENTS ===")
    print(assignments_df.to_string(index=False))

    assignments_df.to_csv("model_a_assignments.csv", index=False)

    # Team travel summary
    team_travel_df = pd.DataFrame({
        "Team": teams,
        "Base City": [team_base_city[t] for t in teams],
        "Total Travel Time (hours)": [team_total[t].X for t in teams]
    }).sort_values("Total Travel Time (hours)", ascending=False)

    print("\n=== TEAM TRAVEL SUMMARY ===")
    print(team_travel_df.to_string(index=False))

    team_travel_df.to_csv("model_a_team_travel.csv", index=False)

    print(f"\nObjective value Z (max team travel): {Z.X:.2f} hours")
    print("\nFiles created:")
    print("- model_a_assignments.csv")
    print("- model_a_team_travel.csv")

else:
    print("No optimal solution found.")

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD EPYC 7763 64-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 84 rows, 349 columns and 900 nonzeros (Min)
Model fingerprint: 0xf08f3ff1
Model has 1 linear objective coefficients
Variable types: 25 continuous, 324 integer (324 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 11.3200000
Presolve removed 14 rows and 193 columns
Presolve time: 0.02s
Presolved: 70 rows, 156 columns, 435 nonzeros
Variable types: 0 continuous, 156 integer (135 binary)

Root relaxation: objective 4.245000e+00, 30 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     

MODEL B: GROUP STAGE

In [ ]:
# ============================================================
# 2) GENERATE ROUND-ROBIN MATCHES
# ============================================================

matches = []
for group_name, team_list in groups.items():
    for k, (t1, t2) in enumerate(itertools.combinations(team_list, 2), start=1):
        matches.append({
            "match_id": f"{group_name}{k}",
            "group": group_name,
            "team1": t1,
            "team2": t2
        })

matches_df = pd.DataFrame(matches)

# ============================================================
# 3) DERIVED STRUCTURES
# ============================================================

teams = sorted(team_base_city.keys())
stadiums = stadiums_df["stadium"].tolist()
cities = sorted(stadiums_df["city"].unique().tolist())

stadium_city = stadiums_df.set_index("stadium")["city"].to_dict()
match_ids = matches_df["match_id"].tolist()
group_of_match = matches_df.set_index("match_id")["group"].to_dict()

# Team-match indicator
a_tm = {}
for row in matches_df.itertuples(index=False):
    for t in teams:
        a_tm[(t, row.match_id)] = int(t == row.team1 or t == row.team2)

# Round-trip travel cost if team t plays match m at stadium s
travel_cost = {}
for row in matches_df.itertuples(index=False):
    m = row.match_id
    for t in teams:
        for s in stadiums:
            if a_tm[(t, m)] == 1:
                base = team_base_city[t]
                venue_city = stadium_city[s]
                travel_cost[(t, m, s)] = 2 * travel_time[base][venue_city]
            else:
                travel_cost[(t, m, s)] = 0.0

# Matches by group
matches_by_group = {
    g: matches_df.loc[matches_df["group"] == g, "match_id"].tolist()
    for g in groups
}

# Morocco matches
morocco_matches = matches_df[
    (matches_df["team1"] == "Morocco") | (matches_df["team2"] == "Morocco")
]["match_id"].tolist()

host_stadium = "Prince Moulay Abdellah Sports Complex"

# ============================================================
# 4) BUILD MODEL B
# ============================================================

model = gp.Model("AFCON_Model_B_Fairness_With_Realism")

# x[m,s] = 1 if match m assigned to stadium s
x = model.addVars(match_ids, stadiums, vtype=GRB.BINARY, name="x")

# y[g,c] = 1 if group g uses city c
y = model.addVars(groups.keys(), cities, vtype=GRB.BINARY, name="y")

# team total travel
team_total = model.addVars(teams, lb=0.0, vtype=GRB.CONTINUOUS, name="team_total")

# maximum team travel
Z = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Z")

# ------------------------------------------------------------
# 5) CORE CONSTRAINTS
# ------------------------------------------------------------

# Each match assigned to exactly one stadium
for m in match_ids:
    model.addConstr(gp.quicksum(x[m, s] for s in stadiums) == 1, name=f"assign_{m}")

# Team total travel
for t in teams:
    model.addConstr(
        team_total[t] ==
        gp.quicksum(travel_cost[(t, m, s)] * x[m, s]
                    for m in match_ids
                    for s in stadiums),
        name=f"team_total_{t}"
    )

# Max fairness
for t in teams:
    model.addConstr(team_total[t] <= Z, name=f"max_travel_{t}")

# ------------------------------------------------------------
# 6) MODEL B REALISM CONSTRAINTS
# ------------------------------------------------------------

# (i) Morocco group-stage matches must be at Prince Moulay Abdellah
for m in morocco_matches:
    model.addConstr(x[m, host_stadium] == 1, name=f"morocco_anchor_{m}")

# (ii) Group locality: each group may use at most 2 cities
for g in groups.keys():
    model.addConstr(gp.quicksum(y[g, c] for c in cities) <= 2, name=f"group_two_cities_{g}")

# Link match-stadium assignments to city-use variables
# If any match of group g is assigned to a stadium in city c, then y[g,c] must be 1
for g in groups.keys():
    for m in matches_by_group[g]:
        for c in cities:
            model.addConstr(
                gp.quicksum(x[m, s] for s in stadiums if stadium_city[s] == c) <= y[g, c],
                name=f"link_{g}_{m}_{c}"
            )

# ------------------------------------------------------------
# 7) OBJECTIVE
# ------------------------------------------------------------

# Stage 1: minimize max team travel
model.setObjective(Z, GRB.MINIMIZE)
model.optimize()

# Stage 2 tie-breaker: among all min-max optimal solutions, minimize total travel
if model.status == GRB.OPTIMAL:
    best_Z = Z.X
    model.addConstr(Z <= best_Z + 1e-6, name="fix_best_Z")
    model.setObjective(gp.quicksum(team_total[t] for t in teams), GRB.MINIMIZE)
    model.optimize()

# ============================================================
# 8) OUTPUT TABLES
# ============================================================

if model.status == GRB.OPTIMAL:
    assignments = []

    for row in matches_df.itertuples(index=False):
        chosen_stadium = None
        for s in stadiums:
            if x[row.match_id, s].X > 0.5:
                chosen_stadium = s
                break

        assignments.append({
            "Match ID": row.match_id,
            "Group": row.group,
            "Team 1": row.team1,
            "Team 2": row.team2,
            "Assigned Stadium": chosen_stadium,
            "Assigned City": stadium_city[chosen_stadium]
        })

    assignments_df = pd.DataFrame(assignments).sort_values(["Group", "Match ID"])
    print("\n=== MODEL B ASSIGNMENTS ===")
    print(assignments_df.to_string(index=False))
    assignments_df.to_csv("model_b_assignments.csv", index=False)

    team_travel_df = pd.DataFrame({
        "Team": teams,
        "Base City": [team_base_city[t] for t in teams],
        "Total Travel Time (hours)": [team_total[t].X for t in teams]
    }).sort_values("Total Travel Time (hours)", ascending=False)

    print("\n=== MODEL B TEAM TRAVEL SUMMARY ===")
    print(team_travel_df.to_string(index=False))
    team_travel_df.to_csv("model_b_team_travel.csv", index=False)

    # Group-city summary
    group_city_rows = []
    for g in groups.keys():
        used_cities = [c for c in cities if y[g, c].X > 0.5]
        group_city_rows.append({
            "Group": g,
            "Used Cities": ", ".join(used_cities),
            "Number of Cities": len(used_cities)
        })

    group_city_df = pd.DataFrame(group_city_rows).sort_values("Group")
    print("\n=== MODEL B GROUP-CITY SUMMARY ===")
    print(group_city_df.to_string(index=False))
    group_city_df.to_csv("model_b_group_cities.csv", index=False)

    print(f"\nObjective value Z (max team travel): {Z.X:.2f} hours")
    print("\nFiles created:")
    print("- model_b_assignments.csv")
    print("- model_b_team_travel.csv")
    print("- model_b_group_cities.csv")

else:
    print("No optimal solution found.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD EPYC 7763 64-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 309 rows, 385 columns and 1479 nonzeros (Min)
Model fingerprint: 0x8848bfaa
Model has 1 linear objective coefficients
Variable types: 25 continuous, 360 integer (360 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Found heuristic solution: objective 16.9800000
Presolve removed 29 rows and 128 columns
Presolve time: 0.00s
Presolved: 280 rows, 257 columns, 1022 nonzeros
Variable types: 0 continuous, 257 integer (233 binary)

Root relaxation: objective 4.245000e+00, 72 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf

MODEL A: ROUND OF 16

In [ ]:
# ============================================================
# 2) ROUND OF 16 MATCHES (8 MATCHES, 16 TEAMS)
# ============================================================

# Assumption: using the actual Round of 16 teams you already defined
matches_df = pd.DataFrame([
    {"match_id": "R16_1", "round": "Round of 16", "team1": "Senegal", "team2": "Sudan"},
    {"match_id": "R16_2", "round": "Round of 16", "team1": "Mali", "team2": "Tunisia"},
    {"match_id": "R16_3", "round": "Round of 16", "team1": "Morocco", "team2": "Tanzania"},
    {"match_id": "R16_4", "round": "Round of 16", "team1": "South Africa", "team2": "Cameroon"},
    {"match_id": "R16_5", "round": "Round of 16", "team1": "Egypt", "team2": "Benin"},
    {"match_id": "R16_6", "round": "Round of 16", "team1": "Nigeria", "team2": "Mozambique"},
    {"match_id": "R16_7", "round": "Round of 16", "team1": "Algeria", "team2": "DR Congo"},
    {"match_id": "R16_8", "round": "Round of 16", "team1": "Cote d'Ivoire", "team2": "Burkina Faso"},
])

# ============================================================
# 3) DERIVED STRUCTURES
# ============================================================

teams_in_knockout = sorted(set(matches_df["team1"]).union(set(matches_df["team2"])))
stadiums = stadiums_df["stadium"].tolist()
stadium_city = stadiums_df.set_index("stadium")["city"].to_dict()
match_ids = matches_df["match_id"].tolist()

# Indicator: a_tm = 1 if team t plays in match m
a_tm = {}
for row in matches_df.itertuples(index=False):
    for t in teams_in_knockout:
        a_tm[(t, row.match_id)] = int(t == row.team1 or t == row.team2)

# Round-trip travel cost for team t if match m is assigned to stadium s
travel_cost = {}
for row in matches_df.itertuples(index=False):
    m = row.match_id
    for t in teams_in_knockout:
        for s in stadiums:
            if a_tm[(t, m)] == 1:
                base = team_base_city[t]
                venue_city = stadium_city[s]
                travel_cost[(t, m, s)] = 2 * travel_time[base][venue_city]
            else:
                travel_cost[(t, m, s)] = 0.0

# ============================================================
# 4) BUILD MODEL A
# ============================================================

model = gp.Model("AFCON_R16_Model_A_Pure_Fairness")

# Decision variable: x[m,s] = 1 if match m assigned to stadium s
x = model.addVars(match_ids, stadiums, vtype=GRB.BINARY, name="x")

# Total travel per team
team_total = model.addVars(teams_in_knockout, lb=0.0, vtype=GRB.CONTINUOUS, name="team_total")

# Maximum team travel
Z = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Z")

# Each match assigned to exactly one stadium
for m in match_ids:
    model.addConstr(
        gp.quicksum(x[m, s] for s in stadiums) == 1,
        name=f"assign_{m}"
    )

# Define team total travel
for t in teams_in_knockout:
    model.addConstr(
        team_total[t] ==
        gp.quicksum(travel_cost[(t, m, s)] * x[m, s]
                    for m in match_ids
                    for s in stadiums),
        name=f"team_total_{t}"
    )

# Max fairness constraint
for t in teams_in_knockout:
    model.addConstr(team_total[t] <= Z, name=f"max_travel_{t}")

# Objective: minimize maximum team travel
model.setObjective(Z, GRB.MINIMIZE)

# ============================================================
# 5) TWO-STAGE SOLVE (OPTIONAL BUT GOOD)
# ============================================================

model.optimize()

if model.status == GRB.OPTIMAL:
    best_Z = Z.X
    model.addConstr(Z <= best_Z + 1e-6, name="fix_best_Z")
    model.setObjective(gp.quicksum(team_total[t] for t in teams_in_knockout), GRB.MINIMIZE)
    model.optimize()

# ============================================================
# 6) OUTPUT ASSIGNMENT TABLE
# ============================================================

if model.status == GRB.OPTIMAL:
    assignments = []

    for row in matches_df.itertuples(index=False):
        chosen_stadium = None
        for s in stadiums:
            if x[row.match_id, s].X > 0.5:
                chosen_stadium = s
                break

        assignments.append({
            "Match ID": row.match_id,
            "Round": row.round,
            "Team 1": row.team1,
            "Team 2": row.team2,
            "Assigned Stadium": chosen_stadium,
            "Assigned City": stadium_city[chosen_stadium]
        })

    assignments_df = pd.DataFrame(assignments)
    print("\n=== ROUND OF 16 MODEL A ASSIGNMENTS ===")
    print(assignments_df.to_string(index=False))
    assignments_df.to_csv("r16_model_a_assignments.csv", index=False)

    team_travel_df = pd.DataFrame({
        "Team": teams_in_knockout,
        "Base City": [team_base_city[t] for t in teams_in_knockout],
        "Total Travel Time (hours)": [team_total[t].X for t in teams_in_knockout]
    }).sort_values("Total Travel Time (hours)", ascending=False)

    print("\n=== ROUND OF 16 MODEL A TEAM TRAVEL ===")
    print(team_travel_df.to_string(index=False))
    team_travel_df.to_csv("r16_model_a_team_travel.csv", index=False)

    print(f"\nObjective value Z (max team travel): {Z.X:.2f} hours")
    print("\nFiles created:")
    print("- r16_model_a_assignments.csv")
    print("- r16_model_a_team_travel.csv")

else:
    print("No optimal solution found.")

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD EPYC 7763 64-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 40 rows, 89 columns and 230 nonzeros (Min)
Model fingerprint: 0xbf9c0b21
Model has 1 linear objective coefficients
Variable types: 17 continuous, 72 integer (72 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 14.5400000
Presolve removed 7 rows and 38 columns
Presolve time: 0.01s
Presolved: 33 rows, 51 columns, 133 nonzeros
Variable types: 0 continuous, 51 integer (37 binary)

Root relaxation: objective 7.270000e+00, 10 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objectiv

MODEL B: R16

In [ ]:
# ============================================================
# 2) ROUND OF 16 MATCHES
# ============================================================

matches_df = pd.DataFrame([
    {"match_id": "R16_1", "round": "Round of 16", "team1": "Senegal", "team2": "Sudan"},
    {"match_id": "R16_2", "round": "Round of 16", "team1": "Mali", "team2": "Tunisia"},
    {"match_id": "R16_3", "round": "Round of 16", "team1": "Morocco", "team2": "Tanzania"},
    {"match_id": "R16_4", "round": "Round of 16", "team1": "South Africa", "team2": "Cameroon"},
    {"match_id": "R16_5", "round": "Round of 16", "team1": "Egypt", "team2": "Benin"},
    {"match_id": "R16_6", "round": "Round of 16", "team1": "Nigeria", "team2": "Mozambique"},
    {"match_id": "R16_7", "round": "Round of 16", "team1": "Algeria", "team2": "DR Congo"},
    {"match_id": "R16_8", "round": "Round of 16", "team1": "Cote d'Ivoire", "team2": "Burkina Faso"},
])

# ============================================================
# 3) DERIVED STRUCTURES
# ============================================================

teams_in_knockout = sorted(set(matches_df["team1"]).union(set(matches_df["team2"])))
stadiums = stadiums_df["stadium"].tolist()
stadium_city = stadiums_df.set_index("stadium")["city"].to_dict()
stadium_capacity = stadiums_df.set_index("stadium")["capacity"].to_dict()
match_ids = matches_df["match_id"].tolist()

# Major venues only for knockout realism
eligible_stadiums = [
    s for s in stadiums if stadium_capacity[s] >= 45000
]

host_stadium = "Prince Moulay Abdellah Sports Complex"

# Indicator: a_tm = 1 if team t plays in match m
a_tm = {}
for row in matches_df.itertuples(index=False):
    for t in teams_in_knockout:
        a_tm[(t, row.match_id)] = int(t == row.team1 or t == row.team2)

# Round-trip travel cost
travel_cost = {}
for row in matches_df.itertuples(index=False):
    m = row.match_id
    for t in teams_in_knockout:
        for s in stadiums:
            if a_tm[(t, m)] == 1:
                base = team_base_city[t]
                venue_city = stadium_city[s]
                travel_cost[(t, m, s)] = 2 * travel_time[base][venue_city]
            else:
                travel_cost[(t, m, s)] = 0.0

# Morocco matches
morocco_matches = matches_df[
    (matches_df["team1"] == "Morocco") | (matches_df["team2"] == "Morocco")
]["match_id"].tolist()

# ============================================================
# 4) BUILD MODEL B
# ============================================================

model = gp.Model("AFCON_R16_Model_B_Fairness_With_Realism")

# x[m,s] = 1 if match m assigned to stadium s
x = model.addVars(match_ids, stadiums, vtype=GRB.BINARY, name="x")

# Team total travel
team_total = model.addVars(teams_in_knockout, lb=0.0, vtype=GRB.CONTINUOUS, name="team_total")

# Maximum team travel
Z = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Z")

# ============================================================
# 5) CORE CONSTRAINTS
# ============================================================

# Each match assigned to exactly one stadium
for m in match_ids:
    model.addConstr(
        gp.quicksum(x[m, s] for s in stadiums) == 1,
        name=f"assign_{m}"
    )

# Team travel totals
for t in teams_in_knockout:
    model.addConstr(
        team_total[t] ==
        gp.quicksum(travel_cost[(t, m, s)] * x[m, s]
                    for m in match_ids
                    for s in stadiums),
        name=f"team_total_{t}"
    )

# Max fairness constraint
for t in teams_in_knockout:
    model.addConstr(team_total[t] <= Z, name=f"max_travel_{t}")

# ============================================================
# 6) MODEL B REALISM CONSTRAINTS
# ============================================================

# (i) Only major venues can be used
for m in match_ids:
    model.addConstr(
        gp.quicksum(x[m, s] for s in eligible_stadiums) == 1,
        name=f"major_venue_only_{m}"
    )

# Equivalent effect: all non-eligible stadiums forced to 0
for m in match_ids:
    for s in stadiums:
        if s not in eligible_stadiums:
            model.addConstr(x[m, s] == 0, name=f"forbid_small_{m}_{s}")

# (ii) Morocco match anchored in Rabat main stadium
for m in morocco_matches:
    model.addConstr(x[m, host_stadium] == 1, name=f"morocco_anchor_{m}")

# (iii) No eligible stadium hosts more than 2 Round of 16 matches
for s in eligible_stadiums:
    model.addConstr(
        gp.quicksum(x[m, s] for m in match_ids) <= 2,
        name=f"stadium_cap_{s}"
    )

# ============================================================
# 7) OBJECTIVE
# ============================================================

# Stage 1: minimize max team travel
model.setObjective(Z, GRB.MINIMIZE)
model.optimize()

# Stage 2: among equally fair solutions, minimize total travel
if model.status == GRB.OPTIMAL:
    best_Z = Z.X
    model.addConstr(Z <= best_Z + 1e-6, name="fix_best_Z")
    model.setObjective(gp.quicksum(team_total[t] for t in teams_in_knockout), GRB.MINIMIZE)
    model.optimize()

# ============================================================
# 8) OUTPUT TABLES
# ============================================================

if model.status == GRB.OPTIMAL:
    assignments = []

    for row in matches_df.itertuples(index=False):
        chosen_stadium = None
        for s in stadiums:
            if x[row.match_id, s].X > 0.5:
                chosen_stadium = s
                break

        assignments.append({
            "Match ID": row.match_id,
            "Round": row.round,
            "Team 1": row.team1,
            "Team 2": row.team2,
            "Assigned Stadium": chosen_stadium,
            "Assigned City": stadium_city[chosen_stadium]
        })

    assignments_df = pd.DataFrame(assignments)
    print("\n=== ROUND OF 16 MODEL B ASSIGNMENTS ===")
    print(assignments_df.to_string(index=False))
    assignments_df.to_csv("r16_model_b_assignments.csv", index=False)

    team_travel_df = pd.DataFrame({
        "Team": teams_in_knockout,
        "Base City": [team_base_city[t] for t in teams_in_knockout],
        "Total Travel Time (hours)": [team_total[t].X for t in teams_in_knockout]
    }).sort_values("Total Travel Time (hours)", ascending=False)

    print("\n=== ROUND OF 16 MODEL B TEAM TRAVEL ===")
    print(team_travel_df.to_string(index=False))
    team_travel_df.to_csv("r16_model_b_team_travel.csv", index=False)

    stadium_usage_rows = []
    for s in eligible_stadiums:
        stadium_usage_rows.append({
            "Stadium": s,
            "City": stadium_city[s],
            "Assigned Matches": sum(1 for m in match_ids if x[m, s].X > 0.5)
        })

    stadium_usage_df = pd.DataFrame(stadium_usage_rows).sort_values(["City", "Stadium"])
    print("\n=== ROUND OF 16 MODEL B STADIUM USAGE ===")
    print(stadium_usage_df.to_string(index=False))
    stadium_usage_df.to_csv("r16_model_b_stadium_usage.csv", index=False)

    print(f"\nObjective value Z (max team travel): {Z.X:.2f} hours")
    print("\nFiles created:")
    print("- r16_model_b_assignments.csv")
    print("- r16_model_b_team_travel.csv")
    print("- r16_model_b_stadium_usage.csv")

else:
    print("No optimal solution found.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD EPYC 7763 64-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 79 rows, 89 columns and 351 nonzeros (Min)
Model fingerprint: 0x078f4864
Model has 1 linear objective coefficients
Variable types: 17 continuous, 72 integer (72 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+00]

Found heuristic solution: objective 14.5400000
Presolve removed 40 rows and 37 columns
Presolve time: 0.00s
Presolved: 39 rows, 52 columns, 172 nonzeros
Variable types: 0 continuous, 52 integer (38 binary)

Root relaxation: objective 7.270000e+00, 15 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumben

MODEL A: Quarter Finals

In [3]:
# ============================================================
# 2) QUARTERFINAL MATCHES
# ============================================================

matches_df = pd.DataFrame([
    {"match_id": "QF_1", "round": "Quarterfinal", "team1": "Mali", "team2": "Senegal"},
    {"match_id": "QF_2", "round": "Quarterfinal", "team1": "Egypt", "team2": "Cote d'Ivoire"},
    {"match_id": "QF_3", "round": "Quarterfinal", "team1": "Cameroon", "team2": "Morocco"},
    {"match_id": "QF_4", "round": "Quarterfinal", "team1": "Algeria", "team2": "Nigeria"},
])

# ============================================================
# 3) DERIVED STRUCTURES
# ============================================================

teams_in_knockout = sorted(set(matches_df["team1"]).union(set(matches_df["team2"])))
stadiums = stadiums_df["stadium"].tolist()
stadium_city = stadiums_df.set_index("stadium")["city"].to_dict()
match_ids = matches_df["match_id"].tolist()

a_tm = {}
for row in matches_df.itertuples(index=False):
    for t in teams_in_knockout:
        a_tm[(t, row.match_id)] = int(t == row.team1 or t == row.team2)

travel_cost = {}
for row in matches_df.itertuples(index=False):
    m = row.match_id
    for t in teams_in_knockout:
        for s in stadiums:
            if a_tm[(t, m)] == 1:
                base = team_base_city[t]
                venue_city = stadium_city[s]
                travel_cost[(t, m, s)] = 2 * travel_time[base][venue_city]
            else:
                travel_cost[(t, m, s)] = 0.0

# ============================================================
# 4) MODEL A
# ============================================================

model = gp.Model("AFCON_QF_Model_A_Pure_Fairness")

x = model.addVars(match_ids, stadiums, vtype=GRB.BINARY, name="x")
team_total = model.addVars(teams_in_knockout, lb=0.0, vtype=GRB.CONTINUOUS, name="team_total")
Z = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Z")

# Each match assigned to exactly one stadium
for m in match_ids:
    model.addConstr(gp.quicksum(x[m, s] for s in stadiums) == 1, name=f"assign_{m}")

# Team total travel
for t in teams_in_knockout:
    model.addConstr(
        team_total[t] ==
        gp.quicksum(travel_cost[(t, m, s)] * x[m, s]
                    for m in match_ids
                    for s in stadiums),
        name=f"team_total_{t}"
    )

# Max fairness
for t in teams_in_knockout:
    model.addConstr(team_total[t] <= Z, name=f"max_travel_{t}")

model.setObjective(Z, GRB.MINIMIZE)
model.optimize()

# Optional second stage: minimize total travel among equally fair solutions
if model.status == GRB.OPTIMAL:
    best_Z = Z.X
    model.addConstr(Z <= best_Z + 1e-6, name="fix_best_Z")
    model.setObjective(gp.quicksum(team_total[t] for t in teams_in_knockout), GRB.MINIMIZE)
    model.optimize()

# ============================================================
# 5) OUTPUT
# ============================================================

if model.status == GRB.OPTIMAL:
    assignments = []
    for row in matches_df.itertuples(index=False):
        chosen_stadium = None
        for s in stadiums:
            if x[row.match_id, s].X > 0.5:
                chosen_stadium = s
                break

        assignments.append({
            "Match ID": row.match_id,
            "Round": row.round,
            "Team 1": row.team1,
            "Team 2": row.team2,
            "Assigned Stadium": chosen_stadium,
            "Assigned City": stadium_city[chosen_stadium]
        })

    assignments_df = pd.DataFrame(assignments)
    print("\n=== QUARTERFINAL MODEL A ASSIGNMENTS ===")
    print(assignments_df.to_string(index=False))
    assignments_df.to_csv("qf_model_a_assignments.csv", index=False)

    team_travel_df = pd.DataFrame({
        "Team": teams_in_knockout,
        "Base City": [team_base_city[t] for t in teams_in_knockout],
        "Total Travel Time (hours)": [team_total[t].X for t in teams_in_knockout]
    }).sort_values("Total Travel Time (hours)", ascending=False)

    print("\n=== QUARTERFINAL MODEL A TEAM TRAVEL ===")
    print(team_travel_df.to_string(index=False))
    team_travel_df.to_csv("qf_model_a_team_travel.csv", index=False)

    print(f"\nObjective value Z (max team travel): {Z.X:.2f} hours")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD EPYC 7763 64-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 20 rows, 45 columns and 118 nonzeros (Min)
Model fingerprint: 0xfdcf22f0
Model has 1 linear objective coefficients
Variable types: 9 continuous, 36 integer (36 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 0 rows and 12 columns
Presolve time: 0.00s
Presolved: 20 rows, 33 columns, 88 nonzeros
Variable types: 0 continuous, 33 integer (24 binary)
Found heuristic solution: objective 11.1400000
Found heuristic solution: objective 9.6000000
Found heuristic solution: objective 7.5400000
Found heuristic solution: objective 7.0000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work uni

MODEL B: Quarter Finals

In [5]:
# ============================================================
# 3) DERIVED STRUCTURES
# ============================================================

teams_in_knockout = sorted(set(matches_df["team1"]).union(set(matches_df["team2"])))
stadiums = stadiums_df["stadium"].tolist()
stadium_city = stadiums_df.set_index("stadium")["city"].to_dict()
stadium_capacity = stadiums_df.set_index("stadium")["capacity"].to_dict()
match_ids = matches_df["match_id"].tolist()

# Major venues only
eligible_stadiums = [s for s in stadiums if stadium_capacity[s] >= 45000]
small_stadiums = [s for s in stadiums if stadium_capacity[s] < 45000]

host_stadium = "Prince Moulay Abdellah Sports Complex"

a_tm = {}
for row in matches_df.itertuples(index=False):
    for t in teams_in_knockout:
        a_tm[(t, row.match_id)] = int(t == row.team1 or t == row.team2)

travel_cost = {}
for row in matches_df.itertuples(index=False):
    m = row.match_id
    for t in teams_in_knockout:
        for s in stadiums:
            if a_tm[(t, m)] == 1:
                base = team_base_city[t]
                venue_city = stadium_city[s]
                travel_cost[(t, m, s)] = 2 * travel_time[base][venue_city]
            else:
                travel_cost[(t, m, s)] = 0.0

morocco_matches = matches_df[
    (matches_df["team1"] == "Morocco") | (matches_df["team2"] == "Morocco")
]["match_id"].tolist()

# ============================================================
# 4) MODEL B
# ============================================================

model = gp.Model("AFCON_QF_Model_B_Fairness_With_Realism")

x = model.addVars(match_ids, stadiums, vtype=GRB.BINARY, name="x")
team_total = model.addVars(teams_in_knockout, lb=0.0, vtype=GRB.CONTINUOUS, name="team_total")
Z = model.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Z")

# Each match assigned to exactly one stadium
for m in match_ids:
    model.addConstr(gp.quicksum(x[m, s] for s in stadiums) == 1, name=f"assign_{m}")

# Team total travel
for t in teams_in_knockout:
    model.addConstr(
        team_total[t] ==
        gp.quicksum(travel_cost[(t, m, s)] * x[m, s]
                    for m in match_ids
                    for s in stadiums),
        name=f"team_total_{t}"
    )

# Max fairness
for t in teams_in_knockout:
    model.addConstr(team_total[t] <= Z, name=f"max_travel_{t}")

# ============================================================
# 5) REALISM CONSTRAINTS
# ============================================================

# Only major venues may host quarterfinals
for m in match_ids:
    model.addConstr(
        gp.quicksum(x[m, s] for s in eligible_stadiums) == 1,
        name=f"major_venue_only_{m}"
    )

for m in match_ids:
    for s in small_stadiums:
        model.addConstr(x[m, s] == 0, name=f"forbid_small_{m}_{s}")

# Morocco anchored in Rabat main stadium
for m in morocco_matches:
    model.addConstr(x[m, host_stadium] == 1, name=f"morocco_anchor_{m}")

# Each eligible stadium hosts at most one quarterfinal
for s in eligible_stadiums:
    model.addConstr(
        gp.quicksum(x[m, s] for m in match_ids) <= 1,
        name=f"stadium_cap_{s}"
    )

# ============================================================
# 6) OBJECTIVE
# ============================================================

model.setObjective(Z, GRB.MINIMIZE)
model.optimize()

# Optional second stage: minimize total travel among equally fair solutions
if model.status == GRB.OPTIMAL:
    best_Z = Z.X
    model.addConstr(Z <= best_Z + 1e-6, name="fix_best_Z")
    model.setObjective(gp.quicksum(team_total[t] for t in teams_in_knockout), GRB.MINIMIZE)
    model.optimize()

# ============================================================
# 7) OUTPUT
# ============================================================

if model.status == GRB.OPTIMAL:
    assignments = []
    for row in matches_df.itertuples(index=False):
        chosen_stadium = None
        for s in stadiums:
            if x[row.match_id, s].X > 0.5:
                chosen_stadium = s
                break

        assignments.append({
            "Match ID": row.match_id,
            "Round": row.round,
            "Team 1": row.team1,
            "Team 2": row.team2,
            "Assigned Stadium": chosen_stadium,
            "Assigned City": stadium_city[chosen_stadium]
        })

    assignments_df = pd.DataFrame(assignments)
    print("\n=== QUARTERFINAL MODEL B ASSIGNMENTS ===")
    print(assignments_df.to_string(index=False))
    assignments_df.to_csv("qf_model_b_assignments.csv", index=False)

    team_travel_df = pd.DataFrame({
        "Team": teams_in_knockout,
        "Base City": [team_base_city[t] for t in teams_in_knockout],
        "Total Travel Time (hours)": [team_total[t].X for t in teams_in_knockout]
    }).sort_values("Total Travel Time (hours)", ascending=False)

    print("\n=== QUARTERFINAL MODEL B TEAM TRAVEL ===")
    print(team_travel_df.to_string(index=False))
    team_travel_df.to_csv("qf_model_b_team_travel.csv", index=False)

    stadium_usage_df = pd.DataFrame([
        {
            "Stadium": s,
            "City": stadium_city[s],
            "Assigned Matches": sum(1 for m in match_ids if x[m, s].X > 0.5)
        }
        for s in eligible_stadiums
    ])

    print("\n=== QUARTERFINAL MODEL B STADIUM USAGE ===")
    print(stadium_usage_df.to_string(index=False))
    stadium_usage_df.to_csv("qf_model_b_stadium_usage.csv", index=False)

    print(f"\nObjective value Z (max team travel): {Z.X:.2f} hours")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD EPYC 7763 64-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 43 rows, 45 columns and 179 nonzeros (Min)
Model fingerprint: 0x73b512fb
Model has 1 linear objective coefficients
Variable types: 9 continuous, 36 integer (36 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 11.1400000
Presolve removed 43 rows and 45 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 2 available processors)

Solution count 2: 11.14 11.14 

Optimal solution found (tolerance 1.00e-04)
Best objective 1.114000000000e+01, best bound 1.114000000000e+01, g

MODEL A: SEMIFINALS

In [10]:
sf_matches_df = pd.DataFrame([
    {"match_id": "SF_1", "round": "Semifinal", "team1": "Senegal", "team2": "Egypt"},
    {"match_id": "SF_2", "round": "Semifinal", "team1": "Morocco", "team2": "Nigeria"},
])

# ------------------------------------------------------------
# 2) DERIVED STRUCTURES
# ------------------------------------------------------------
sf_teams = sorted(set(sf_matches_df["team1"]).union(set(sf_matches_df["team2"])))
sf_stadiums = stadiums_df["stadium"].tolist()
sf_stadium_city = stadiums_df.set_index("stadium")["city"].to_dict()
sf_match_ids = sf_matches_df["match_id"].tolist()

# a_tm = 1 if team t plays in match m
sf_a_tm = {}
for row in sf_matches_df.itertuples(index=False):
    for t in sf_teams:
        sf_a_tm[(t, row.match_id)] = int(t == row.team1 or t == row.team2)

# Round-trip travel cost for team t if match m is assigned to stadium s
sf_travel_cost = {}
for row in sf_matches_df.itertuples(index=False):
    m = row.match_id
    for t in sf_teams:
        for s in sf_stadiums:
            if sf_a_tm[(t, m)] == 1:
                base = team_base_city[t]
                venue_city = sf_stadium_city[s]
                sf_travel_cost[(t, m, s)] = 2 * travel_time[base][venue_city]
            else:
                sf_travel_cost[(t, m, s)] = 0.0

# ------------------------------------------------------------
# 3) BUILD MODEL A
# ------------------------------------------------------------
sf_model_a = gp.Model("AFCON_SF_Model_A_Pure_Fairness")

# Decision variables
sf_x_a = sf_model_a.addVars(sf_match_ids, sf_stadiums, vtype=GRB.BINARY, name="x_sf_a")
sf_team_total_a = sf_model_a.addVars(sf_teams, lb=0.0, vtype=GRB.CONTINUOUS, name="team_total_sf_a")
sf_Z_a = sf_model_a.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Z_sf_a")

# Each match assigned to exactly one stadium
for m in sf_match_ids:
    sf_model_a.addConstr(
        gp.quicksum(sf_x_a[m, s] for s in sf_stadiums) == 1,
        name=f"assign_sf_a_{m}"
    )

# Team total travel
for t in sf_teams:
    sf_model_a.addConstr(
        sf_team_total_a[t] ==
        gp.quicksum(
            sf_travel_cost[(t, m, s)] * sf_x_a[m, s]
            for m in sf_match_ids
            for s in sf_stadiums
        ),
        name=f"team_total_sf_a_{t}"
    )

# Max fairness constraints
for t in sf_teams:
    sf_model_a.addConstr(
        sf_team_total_a[t] <= sf_Z_a,
        name=f"max_travel_sf_a_{t}"
    )

# Stage 1: minimize maximum team travel
sf_model_a.setObjective(sf_Z_a, GRB.MINIMIZE)
sf_model_a.optimize()

# Stage 2: among equally fair solutions, minimize total travel
if sf_model_a.SolCount > 0:
    best_Z_sf_a = sf_Z_a.X
    sf_model_a.addConstr(sf_Z_a <= best_Z_sf_a + 1e-6, name="fix_best_Z_sf_a")
    sf_model_a.setObjective(gp.quicksum(sf_team_total_a[t] for t in sf_teams), GRB.MINIMIZE)
    sf_model_a.optimize()

# ------------------------------------------------------------
# 4) OUTPUT
# ------------------------------------------------------------

print("Model status code:", sf_model_a.Status)
print("Number of solutions found:", sf_model_a.SolCount)

if sf_model_a.SolCount > 0:
    sf_assignments_a = []

    for row in sf_matches_df.itertuples(index=False):
        chosen_stadium = None
        for s in sf_stadiums:
            if sf_x_a[row.match_id, s].X > 0.5:
                chosen_stadium = s
                break

        sf_assignments_a.append({
            "Match ID": row.match_id,
            "Round": row.round,
            "Team 1": row.team1,
            "Team 2": row.team2,
            "Assigned Stadium": chosen_stadium,
            "Assigned City": sf_stadium_city[chosen_stadium]
        })

    sf_assignments_df_a = pd.DataFrame(sf_assignments_a)

    sf_team_travel_df_a = pd.DataFrame({
        "Team": sf_teams,
        "Base City": [team_base_city[t] for t in sf_teams],
        "Total Travel Time (hours)": [sf_team_total_a[t].X for t in sf_teams]
    }).sort_values("Total Travel Time (hours)", ascending=False)

    print("\n=== SEMIFINAL MODEL A ASSIGNMENTS ===")
    print(sf_assignments_df_a.to_string(index=False))

    print("\n=== SEMIFINAL MODEL A TEAM TRAVEL ===")
    print(sf_team_travel_df_a.to_string(index=False))

    assignments_file = "sf_model_a_assignments.csv"
    travel_file = "sf_model_a_team_travel.csv"

    sf_assignments_df_a.to_csv(assignments_file, index=False)
    sf_team_travel_df_a.to_csv(travel_file, index=False)

    print(f"\nObjective value Z (max team travel): {sf_Z_a.X:.2f} hours")
    print("\nFiles created:")
    print(f"- {assignments_file}")
    print(f"- {travel_file}")

else:
    print("No solution found for Semifinal Model A.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD EPYC 7763 64-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 10 rows, 23 columns and 59 nonzeros (Min)
Model fingerprint: 0x6e0dc365
Model has 1 linear objective coefficients
Variable types: 5 continuous, 18 integer (18 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 0 rows and 6 columns
Presolve time: 0.00s
Presolved: 10 rows, 17 columns, 44 nonzeros
Variable types: 0 continuous, 17 integer (12 binary)
Found heuristic solution: objective 15.7400000
Found heuristic solution: objective 9.6000000

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 2 (of 2 available processors)

Solution count 2: 9.6 15.74 

Optimal solu

MODEL B: SEMIFINALS

In [8]:
sf_model_b = gp.Model("AFCON_SF_Model_B_Fairness_With_Realism")

# Decision variables
sf_x_b = sf_model_b.addVars(sf_match_ids, sf_stadiums, vtype=GRB.BINARY, name="x_sf_b")
sf_team_total_b = sf_model_b.addVars(sf_teams, lb=0.0, vtype=GRB.CONTINUOUS, name="team_total_sf_b")
sf_Z_b = sf_model_b.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="Z_sf_b")

# Each match assigned to exactly one stadium
for m in sf_match_ids:
    sf_model_b.addConstr(
        gp.quicksum(sf_x_b[m, s] for s in sf_stadiums) == 1,
        name=f"assign_sf_b_{m}"
    )

# Team total travel
for t in sf_teams:
    sf_model_b.addConstr(
        sf_team_total_b[t] ==
        gp.quicksum(sf_travel_cost[(t, m, s)] * sf_x_b[m, s]
                    for m in sf_match_ids
                    for s in sf_stadiums),
        name=f"team_total_sf_b_{t}"
    )

# Max fairness
for t in sf_teams:
    sf_model_b.addConstr(sf_team_total_b[t] <= sf_Z_b, name=f"max_travel_sf_b_{t}")

# ------------------------------------------------------------
# Realism constraints
# ------------------------------------------------------------

# Only major venues may host semifinals
for m in sf_match_ids:
    sf_model_b.addConstr(
        gp.quicksum(sf_x_b[m, s] for s in sf_eligible_stadiums) == 1,
        name=f"major_venue_only_sf_b_{m}"
    )

for m in sf_match_ids:
    for s in sf_small_stadiums:
        sf_model_b.addConstr(sf_x_b[m, s] == 0, name=f"forbid_small_sf_b_{m}_{s}")

# Morocco semifinal anchored in Rabat main stadium
for m in sf_morocco_matches:
    sf_model_b.addConstr(sf_x_b[m, sf_host_stadium] == 1, name=f"morocco_anchor_sf_b_{m}")

# Each eligible stadium hosts at most one semifinal
for s in sf_eligible_stadiums:
    sf_model_b.addConstr(
        gp.quicksum(sf_x_b[m, s] for m in sf_match_ids) <= 1,
        name=f"stadium_cap_sf_b_{s}"
    )

# Objective
sf_model_b.setObjective(sf_Z_b, GRB.MINIMIZE)
sf_model_b.optimize()

# Optional second stage: minimize total travel among equally fair solutions
if sf_model_b.status == GRB.OPTIMAL:
    best_Z_sf_b = sf_Z_b.X
    sf_model_b.addConstr(sf_Z_b <= best_Z_sf_b + 1e-6, name="fix_best_Z_sf_b")
    sf_model_b.setObjective(gp.quicksum(sf_team_total_b[t] for t in sf_teams), GRB.MINIMIZE)
    sf_model_b.optimize()

# Output
if sf_model_b.status == GRB.OPTIMAL:
    sf_assignments_b = []

    for row in sf_matches_df.itertuples(index=False):
        chosen_stadium = None
        for s in sf_stadiums:
            if sf_x_b[row.match_id, s].X > 0.5:
                chosen_stadium = s
                break

        sf_assignments_b.append({
            "Match ID": row.match_id,
            "Round": row.round,
            "Team 1": row.team1,
            "Team 2": row.team2,
            "Assigned Stadium": chosen_stadium,
            "Assigned City": sf_stadium_city[chosen_stadium]
        })

    sf_assignments_df_b = pd.DataFrame(sf_assignments_b)
    print("\n=== SEMIFINAL MODEL B ASSIGNMENTS ===")
    print(sf_assignments_df_b.to_string(index=False))
    sf_assignments_df_b.to_csv("sf_model_b_assignments.csv", index=False)

    sf_team_travel_df_b = pd.DataFrame({
        "Team": sf_teams,
        "Base City": [team_base_city[t] for t in sf_teams],
        "Total Travel Time (hours)": [sf_team_total_b[t].X for t in sf_teams]
    }).sort_values("Total Travel Time (hours)", ascending=False)

    print("\n=== SEMIFINAL MODEL B TEAM TRAVEL ===")
    print(sf_team_travel_df_b.to_string(index=False))
    sf_team_travel_df_b.to_csv("sf_model_b_team_travel.csv", index=False)

    sf_stadium_usage_df_b = pd.DataFrame([
        {
            "Stadium": s,
            "City": sf_stadium_city[s],
            "Assigned Matches": sum(1 for m in sf_match_ids if sf_x_b[m, s].X > 0.5)
        }
        for s in sf_eligible_stadiums
    ])

    print("\n=== SEMIFINAL MODEL B STADIUM USAGE ===")
    print(sf_stadium_usage_df_b.to_string(index=False))
    sf_stadium_usage_df_b.to_csv("sf_model_b_stadium_usage.csv", index=False)

    print(f"\nObjective value Z (max team travel): {sf_Z_b.X:.2f} hours")
else:
    print("No optimal solution found for Semifinal Model B.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: AMD EPYC 7763 64-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 25 rows, 23 columns and 90 nonzeros (Min)
Model fingerprint: 0x3105d2c8
Model has 1 linear objective coefficients
Variable types: 5 continuous, 18 integer (18 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 15.7400000
Presolve removed 20 rows and 15 columns
Presolve time: 0.00s
Presolved: 5 rows, 8 columns, 19 nonzeros
Variable types: 0 continuous, 8 integer (5 binary)

Root relaxation: objective 7.870000e+00, 3 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    Bes